# 实验五：必须把整张图传过去吗？
## 面向任务的语义通信 · Task-Oriented Semantic Communication under Noisy Channels

**课程**：未来媒体互联网（Future Media & Internet） &nbsp;|&nbsp; **预计时长**：~12–15 分钟  
**运行环境**：Kaggle Notebook（CPU） &nbsp;|&nbsp; **无需 GPU** &nbsp;|&nbsp; **Internet Off**

---

### 一句话问题

> **如果接收端只需要知道“这是数字几”，还有必要完整恢复每一个像素吗？**

Demo4 关注：**有限 Rate 下怎样尽量恢复原始信号？**  
Demo5 进一步追问：**如果最终只需要完成任务，通信系统能否只传任务真正需要的信息？**


## 学习目标

完成本实验后，你应该能够：

1. 区分 **Signal-Oriented Communication** 与 **Task-Oriented Communication**；
2. 理解公平比较为什么必须统一 **Channel Uses、平均发送功率和 SNR**；
3. 理解 **Encoder → Power Normalization → AWGN → Decoder / Task Head**；
4. 比较重建导向和任务导向系统的 **Accuracy vs SNR**；
5. 理解 Signal Metric（PSNR）与 Task Metric（Accuracy）并不等价；
6. 观察两种 16 维 channel representation 的组织方式；
7. 用低 SNR Confusion Matrix 分析失败模式；
8. 准确理解语义通信目前的定位：面向未来网络与 6G 的活跃研究和标准化方向之一。


## Demo4 → Demo5：从“保住信号”到“保住任务”

### Reconstruction-Oriented
**Image → 16 channel symbols → AWGN → reconstructed image → classifier**

目标：尽量恢复图像。

### Task-Oriented Semantic
**Image → 16 channel symbols → AWGN → class decision**

目标：直接完成数字分类。

两种系统都只能使用：

- **16 个 real-valued channel symbols**
- **相同平均发送功率**
- **相同 AWGN**
- **相同 SNR**

因此真正比较的是：

> **Same channel budget. Different communication objective.**


## 运行环境

| 项目 | 设置 |
|---|---|
| 平台 | Kaggle Notebook |
| 计算资源 | CPU |
| GPU | 不需要 |
| Internet | Off |
| 数据集 | `sklearn.datasets.load_digits()` |
| 样本数 | 1797 |
| 图像尺寸 | 8×8 grayscale |
| 类别 | 0–9 |
| Channel Uses | 16 real symbols |
| 训练信道 | SNR 随机采样于 -5 到 10 dB |
| 测试信道 | 10 / 5 / 0 / -5 / -10 dB |

直接 **Run All** 即可。


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

CHANNEL_USES = 16
TRAIN_SNR_MIN = -5.0
TRAIN_SNR_MAX = 10.0

digits = load_digits()
X = (digits.data / 16.0).astype(np.float32)
y = digits.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)

print("Environment ready")
print(f"Dataset: {len(X)} | train={len(X_train)} | test={len(X_test)}")
print(f"Image: 8x8 | classes=10 | channel uses={CHANNEL_USES}")
print(f"Training SNR range: {TRAIN_SNR_MIN:g} to {TRAIN_SNR_MAX:g} dB")


# 第一幕：先看数据，但不要把“低维”自动叫作“语义”

`load_digits()` 是 scikit-learn 内置数据集，不需要联网。

这里研究的是：

> **“通信目标”如何改变表示，而不是模型规模。**

需要特别注意：

> **一个 16 维 latent 不会因为维度低就自动成为 semantic representation。**

只有当训练目标明确要求它保留**任务相关信息**时，才更适合称为 task-oriented semantic representation。


In [2]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.ravel()):
    idx = np.where(y_train == digit)[0][0]
    ax.imshow(X_train[idx].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Digit {digit}")
    ax.axis("off")
plt.suptitle("Offline Digits Dataset")
plt.tight_layout()
plt.show()


# 第二幕：统一物理信道

两种 Encoder 都输出 **16 个 real-valued channel symbols**。

每个样本做功率归一化：

\[
\frac{1}{K}\|z\|^2=1
\]

其中 \(K=16\)。

AWGN：

\[
y=z+n
\]

归一化后：

\[
\sigma_n^2=10^{-SNR_{dB}/10}
\]

因此两种系统共享：

**Same bandwidth + Same power + Same channel**


In [3]:
def power_normalize(z, eps=1e-8):
    power = z.pow(2).mean(dim=1, keepdim=True)
    return z / torch.sqrt(power + eps)

def awgn_channel(z, snr_db):
    # z is normalized to unit average symbol power per sample.
    if not torch.is_tensor(snr_db):
        snr_db = torch.full(
            (z.size(0), 1), float(snr_db),
            dtype=z.dtype, device=z.device
        )
    elif snr_db.ndim == 0:
        snr_db = snr_db.expand(z.size(0)).reshape(-1, 1)
    elif snr_db.ndim == 1:
        snr_db = snr_db.reshape(-1, 1)

    noise_std = torch.sqrt(1.0 / (10.0 ** (snr_db / 10.0)))
    return z + torch.randn_like(z) * noise_std

def sample_training_snr(batch_size):
    return torch.empty(batch_size).uniform_(TRAIN_SNR_MIN, TRAIN_SNR_MAX)

print("Shared AWGN channel ready")


# 第三幕：训练固定的接收端分类器

为了评价“重建后的图还能不能完成任务”，先训练一个普通数字分类器。

它只在**干净训练图像**上训练，随后冻结：

> **Reconstruction system → reconstructed image → fixed classifier → accuracy**


In [4]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 96), nn.ReLU(),
            nn.Linear(96, 48), nn.ReLU(),
            nn.Linear(48, 10)
        )
    def forward(self, x):
        return self.net(x)

receiver_classifier = DigitClassifier()
opt = torch.optim.Adam(receiver_classifier.parameters(), lr=2e-3)

CLASSIFIER_EPOCHS = 60
classifier_history = []

for epoch in range(CLASSIFIER_EPOCHS):
    perm = torch.randperm(len(X_train_t))
    total = 0.0
    for start in range(0, len(perm), 128):
        idx = perm[start:start+128]
        logits = receiver_classifier(X_train_t[idx])
        loss = F.cross_entropy(logits, y_train_t[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)
    classifier_history.append(total / len(X_train_t))

receiver_classifier.eval()
with torch.no_grad():
    clean_accuracy = (
        receiver_classifier(X_test_t).argmax(1) == y_test_t
    ).float().mean().item()

print(f"Clean receiver-classifier accuracy: {clean_accuracy*100:.2f}%")


# 第四幕：两种系统，只改变“通信目标”

## A. Reconstruction-Oriented

Encoder 和 Decoder 通过 MSE 训练：

\[
L_{recon}=MSE(x,\hat{x})
\]

目标：**尽量恢复原图。**

## B. Task-Oriented Semantic

Encoder 后直接接任务分类头：

\[
L_{task}=CrossEntropy(y,\hat y)
\]

目标：**不恢复图像，直接把类别判断正确。**

两套 Encoder 都是：

**64 → 64 → 16 channel symbols**


In [5]:
class ReconstructionSystem(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, CHANNEL_USES)
        )
        self.decoder = nn.Sequential(
            nn.Linear(CHANNEL_USES, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.Sigmoid()
        )

    def forward(self, x, snr_db):
        z = power_normalize(self.encoder(x))
        y_channel = awgn_channel(z, snr_db)
        return self.decoder(y_channel), z

class TaskSemanticSystem(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, CHANNEL_USES)
        )
        self.task_head = nn.Sequential(
            nn.Linear(CHANNEL_USES, 48), nn.ReLU(),
            nn.Linear(48, 10)
        )

    def forward(self, x, snr_db):
        z = power_normalize(self.encoder(x))
        y_channel = awgn_channel(z, snr_db)
        return self.task_head(y_channel), z

reconstruction_system = ReconstructionSystem()
semantic_system = TaskSemanticSystem()

print("Reconstruction params:",
      sum(p.numel() for p in reconstruction_system.parameters()))
print("Task-semantic params:",
      sum(p.numel() for p in semantic_system.parameters()))


# 第五幕：在相同 SNR 分布上训练

原版只在固定 10 dB 训练，再直接测试到 -10 dB。

新版每个 batch 都随机采样：

> **SNR ∼ Uniform(-5, 10) dB**

两套系统看到完全相同的训练信道分布。

测试再加入 **-10 dB stress test**，观察训练范围之外的退化。


In [6]:
def train_reconstruction(model, epochs=100):
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    history = []
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(X_train_t))
        total = 0.0
        for start in range(0, len(perm), 128):
            idx = perm[start:start+128]
            snr = sample_training_snr(len(idx))
            rec, _ = model(X_train_t[idx], snr)
            loss = F.mse_loss(rec, X_train_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        history.append(total / len(X_train_t))
    return history

def train_semantic(model, epochs=80):
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    history = []
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(len(X_train_t))
        total = 0.0
        for start in range(0, len(perm), 128):
            idx = perm[start:start+128]
            snr = sample_training_snr(len(idx))
            logits, _ = model(X_train_t[idx], snr)
            loss = F.cross_entropy(logits, y_train_t[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        history.append(total / len(X_train_t))
    return history

recon_history = train_reconstruction(reconstruction_system, 100)
semantic_history = train_semantic(semantic_system, 80)

print(
    f"Training complete | reconstruction MSE={recon_history[-1]:.4f} | "
    f"semantic CE={semantic_history[-1]:.4f}"
)

plt.figure(figsize=(8,4))
plt.plot(recon_history)
plt.xlabel("Epoch"); plt.ylabel("MSE")
plt.title("Reconstruction-Oriented Training")
plt.grid(alpha=0.2); plt.show()

plt.figure(figsize=(8,4))
plt.plot(semantic_history)
plt.xlabel("Epoch"); plt.ylabel("Cross-Entropy")
plt.title("Task-Oriented Semantic Training")
plt.grid(alpha=0.2); plt.show()


# 第六幕：主实验 —— Accuracy vs SNR

每个 SNR 下：

- 两种系统都使用 **16 channel symbols**；
- 都做单位平均功率归一化；
- 都使用同一个 AWGN 定义；
- 每个 SNR 重复多次独立噪声试验；
- 报告平均 Accuracy 和标准差。

核心指标从“图片看起来像不像”改成：

> **接收端任务有没有完成？**


In [7]:
SNR_LEVELS = [10, 5, 0, -5, -10]
MONTE_CARLO_TRIALS = 12

reconstruction_system.eval()
semantic_system.eval()
receiver_classifier.eval()

rows = []

for snr in SNR_LEVELS:
    ra, ta, rp = [], [], []

    for trial in range(MONTE_CARLO_TRIALS):
        seed = SEED + 1000*(snr+20) + trial

        torch.manual_seed(seed)
        with torch.no_grad():
            rec, _ = reconstruction_system(X_test_t, snr)
            pred = receiver_classifier(rec).argmax(1)
            ra.append((pred == y_test_t).float().mean().item())
            mse = F.mse_loss(rec, X_test_t).item()
            rp.append(10*np.log10(1/max(mse,1e-12)))

        torch.manual_seed(seed + 500000)
        with torch.no_grad():
            logits, _ = semantic_system(X_test_t, snr)
            pred = logits.argmax(1)
            ta.append((pred == y_test_t).float().mean().item())

    rows.append({
        "SNR (dB)": snr,
        "Recon Accuracy": np.mean(ra),
        "Recon Accuracy Std": np.std(ra),
        "Task Accuracy": np.mean(ta),
        "Task Accuracy Std": np.std(ta),
        "Recon PSNR (dB)": np.mean(rp),
    })

results_df = pd.DataFrame(rows)
display(results_df.style.format({
    "Recon Accuracy": "{:.3f}",
    "Recon Accuracy Std": "{:.3f}",
    "Task Accuracy": "{:.3f}",
    "Task Accuracy Std": "{:.3f}",
    "Recon PSNR (dB)": "{:.2f}",
}))

plt.figure(figsize=(9,5))
plt.errorbar(
    results_df["SNR (dB)"], results_df["Recon Accuracy"]*100,
    yerr=results_df["Recon Accuracy Std"]*100,
    marker="o", capsize=4, label="Reconstruction-oriented -> classifier"
)
plt.errorbar(
    results_df["SNR (dB)"], results_df["Task Accuracy"]*100,
    yerr=results_df["Task Accuracy Std"]*100,
    marker="o", capsize=4, label="Task-oriented semantic"
)
plt.xlabel("Channel SNR (dB)")
plt.ylabel("Digit Classification Accuracy (%)")
plt.title("Same Channel Budget: Task Accuracy vs SNR")
plt.ylim(0,102)
plt.grid(alpha=0.2)
plt.legend()
plt.show()


# 第七幕：同一个样本，两种系统到底交付什么？

Reconstruction-Oriented：

> **先恢复图像，再让下游分类器识别。**

Task-Oriented Semantic：

> **不恢复图像，直接交付任务答案。**

后者没有 reconstructed image，因此 PSNR 并不是它的正确主指标。


In [8]:
DEMO_SNR = -5
demo_idx = int(np.where(y_test == 8)[0][0])
demo_x = X_test_t[demo_idx:demo_idx+1]
demo_y = int(y_test_t[demo_idx])

torch.manual_seed(12345)
with torch.no_grad():
    demo_rec, _ = reconstruction_system(demo_x, DEMO_SNR)
    rec_pred = int(receiver_classifier(demo_rec).argmax(1).item())

torch.manual_seed(54321)
with torch.no_grad():
    task_logits, _ = semantic_system(demo_x, DEMO_SNR)
    task_probs = torch.softmax(task_logits,1).squeeze().numpy()
    task_pred = int(task_logits.argmax(1).item())

plt.figure(figsize=(4,4))
plt.imshow(demo_x.squeeze().numpy().reshape(8,8), cmap="gray", vmin=0, vmax=1)
plt.title(f"Original | true digit = {demo_y}")
plt.axis("off"); plt.show()

plt.figure(figsize=(4,4))
plt.imshow(demo_rec.squeeze().numpy().reshape(8,8), cmap="gray", vmin=0, vmax=1)
plt.title(f"Reconstruction @ {DEMO_SNR} dB | prediction={rec_pred}")
plt.axis("off"); plt.show()

plt.figure(figsize=(8,4))
plt.bar(np.arange(10), task_probs)
plt.xticks(np.arange(10)); plt.ylim(0,1)
plt.xlabel("Digit class"); plt.ylabel("Probability")
plt.title(f"Task-oriented semantic @ {DEMO_SNR} dB | direct prediction={task_pred}")
plt.show()

print(
    f"True label={demo_y} | reconstruction-path prediction={rec_pred} | "
    f"task-semantic prediction={task_pred}"
)


# 第八幕：16 个 channel symbols 学到了什么？

用 PCA **只做二维可视化**：

- Reconstruction-Oriented representation 首先服务于像素重建；
- Task-Oriented representation 直接服务于数字类别。

如果 task-oriented latent 的类别聚类更清晰，就说明：

> **训练目标正在塑造“什么信息值得进入信道”。**


In [9]:
with torch.no_grad():
    recon_latent = power_normalize(
        reconstruction_system.encoder(X_test_t)
    ).numpy()
    semantic_latent = power_normalize(
        semantic_system.encoder(X_test_t)
    ).numpy()

recon_2d = PCA(n_components=2, random_state=SEED).fit_transform(recon_latent)
semantic_2d = PCA(n_components=2, random_state=SEED).fit_transform(semantic_latent)

plt.figure(figsize=(8,6))
for digit in range(10):
    mask = y_test == digit
    plt.scatter(recon_2d[mask,0], recon_2d[mask,1], s=18, alpha=0.65, label=str(digit))
plt.xlabel("PCA-1"); plt.ylabel("PCA-2")
plt.title("Reconstruction-Oriented Channel Representation")
plt.legend(ncol=5, title="Digit"); plt.grid(alpha=0.15); plt.show()

plt.figure(figsize=(8,6))
for digit in range(10):
    mask = y_test == digit
    plt.scatter(semantic_2d[mask,0], semantic_2d[mask,1], s=18, alpha=0.65, label=str(digit))
plt.xlabel("PCA-1"); plt.ylabel("PCA-2")
plt.title("Task-Oriented Semantic Channel Representation")
plt.legend(ncol=5, title="Digit"); plt.grid(alpha=0.15); plt.show()


# 第九幕：低 SNR 时到底错在哪里？

下面固定 **-5 dB** 画 Confusion Matrix。

总 Accuracy 告诉我们“错了多少”，Confusion Matrix 告诉我们：

> **哪些数字更容易互相混淆？**


In [10]:
CONFUSION_SNR = -5

torch.manual_seed(777)
with torch.no_grad():
    rec_low, _ = reconstruction_system(X_test_t, CONFUSION_SNR)
    rec_pred = receiver_classifier(rec_low).argmax(1).numpy()

torch.manual_seed(888)
with torch.no_grad():
    sem_logits, _ = semantic_system(X_test_t, CONFUSION_SNR)
    sem_pred = sem_logits.argmax(1).numpy()

cm_rec = confusion_matrix(y_test, rec_pred, labels=np.arange(10))
cm_sem = confusion_matrix(y_test, sem_pred, labels=np.arange(10))

plt.figure(figsize=(7,6))
plt.imshow(cm_rec)
plt.title(f"Reconstruction-Oriented Confusion Matrix @ {CONFUSION_SNR} dB")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(np.arange(10)); plt.yticks(np.arange(10)); plt.colorbar()
for i in range(10):
    for j in range(10):
        if cm_rec[i,j] > 0:
            plt.text(j,i,str(cm_rec[i,j]),ha="center",va="center",fontsize=7)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,6))
plt.imshow(cm_sem)
plt.title(f"Task-Oriented Semantic Confusion Matrix @ {CONFUSION_SNR} dB")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(np.arange(10)); plt.yticks(np.arange(10)); plt.colorbar()
for i in range(10):
    for j in range(10):
        if cm_sem[i,j] > 0:
            plt.text(j,i,str(cm_sem[i,j]),ha="center",va="center",fontsize=7)
plt.tight_layout(); plt.show()


# 实验结果应该怎样解读？

新版不再声称：

> “Semantic communication 的图片在低 SNR 下更清楚，所以一定更好。”

更准确的结论是：

1. **通信目标决定表示**：重建系统保留像素信息；任务系统保留分类相关信息。
2. **Channel budget 没变，Objective 变了**：两种系统都只有 16 个 channel symbols。
3. **Signal Metric 与 Task Metric 不等价**：PSNR 更适合评价重建，Accuracy 更适合评价任务。
4. **本实验不再声称验证传统通信 cliff effect**：原版“像素直接加 AWGN”并没有实现 source coding、channel coding、modulation 和 hard decoding。

因此，本实验真正比较的是：

> **Signal-oriented objective vs Task-oriented objective under the same learned analog channel budget.**


# 从课堂实验到真实研究

### DeepJSCC / learned joint source-channel coding
Encoder–Channel–Decoder 可以针对信道联合优化。以图像重建为目标的 DeepJSCC 类方法仍然主要属于 **signal reconstruction-oriented** 路线。

### Task-Oriented Semantic Communication
更进一步的系统可以直接优化分类、检测、控制等下游任务，而不要求恢复全部输入。

### 标准化状态要准确表述
截至 2026 年，语义通信更适合表述为：

> **面向未来网络与 6G 的活跃研究和标准化方向之一。**

ITU-T 当前有多项相关工作仍处于 **Under study**，包括 knowledge-based semantic communication、IoT semantic communication、AI-based semantic video calls 等。

延伸阅读：

- https://www.itu.int/itu-t/workprog/wp_item.aspx?isn=23408
- https://www.itu.int/itu-t/workprog/wp_item.aspx?isn=21985
- https://www.itu.int/ITU-T/workprog/wp_item.aspx?isn=23625


## 实验局限性与思考

### 有意简化
1. 数据集只有 8×8 手写数字；
2. 信道只有 AWGN；
3. channel symbols 是 real-valued learned symbols，不是实际调制波形；
4. 统一了 Channel Uses / Power / SNR，但没有强制完全相同参数量；
5. 没有真实 source coding / channel coding baseline；
6. 没有带宽、时延、能耗系统级分析；
7. -10 dB 超出训练 SNR 下界，是 stress test。

### 思考题
1. 把 `CHANNEL_USES=16` 改成 4，哪个系统退化更快？
2. 如果任务改成“精确恢复笔画”，Task-Oriented 还会占优吗？
3. 为什么没有 reconstructed image 时，PSNR 不是主指标？
4. 同时优化 MSE + Classification Loss 会怎样？
5. 换成 Rayleigh fading，需要改什么？
6. 能否让 Encoder 根据 SNR 动态改变表示？
7. 如果只发送类别标签，为什么还需要 16 个 learned symbols？这与可靠数字通信有什么区别？
8. 怎样加入真实 channel coding baseline，再研究 cliff effect？


---

← [实验四：神经压缩 vs JPEG](https://www.kaggle.com/code/guopingtan/fmi-demo4-neural-compression)
&nbsp;|&nbsp;
🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here)
&nbsp;|&nbsp;
[实验六：网络损伤对高清视频的影响 →](https://www.kaggle.com/code/guopingtan/fmi-demo-6-network-impairments-on-hd-video)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University
